Stamp – 2026-07-29 | dlnm-pilot | code

Goal
Put a number on the flat DA map.

Produced
Number: recovered SD exactly 0 across 7,682 DAs, 1 distinct value, identical(sd(x), 0) TRUE.
rr_flat 3.176 at the 99th percentile (27.19°C), MMT 21.68°C, cold end RR 81.7.
§8.8 predicts through the reduced basis with model.link = "log" declared and the five (Intercept) terms sliced by name.
Six SSOT edits: the link, two object renames, the intercept slice, the v1 correction, the v2 baseline, the save_to_drive note.
Toronto truth_factors + temp_mat at seed 77, μ [−0.33, 0.655, 0.384], truth SD 0.4454, mean 0.946.

Missed
save_to_drive tars the directory and never empties it. Opened: 141 MB of the 178.5 MB tarball is cross-basis, and §8 uses argvar alone.
Predicted rr_flat 5–20, got 3.176 — put the last basis function near 0.5 mid-span when a quadratic rises as a square and sits near 0.25, so θ₅ = −1.02 got over-weighted.
Predicted manual vs crosspred at floating point, got 0.298 — i_cen takes the nearest grid point (19.422), crosspred centres at 19.38.
Predicted mtl_substrate 8–15 MB, got 20.26 — gzip caps near 2× on doubles.
Predicted RR at cen exactly 1, got 1.000241.

Next
Project Toronto's 7,682 × 17 Z through pca_cma with predict.prcomp and correlate those scores against fit_da_pca's.

§0 restore into a cold kernel. library cache, package stack, pilot_session for the dgp
primitives, then source fns.R after load() unpack saves_eod_2026-07-22, read the eight rds back.

In [ ]:
DRIVE <- "/content/drive/MyDrive/thesis/dlnm-pilot"
stopifnot(dir.exists(DRIVE))

system(sprintf("cd /content && cp %s/r_library.tar.gz . && tar -xzf r_library.tar.gz", DRIVE))
.libPaths(c("/content/site-library", .libPaths()))

suppressPackageStartupMessages({
  library(dlnm); library(gnm); library(mixmeta); library(splines)
  library(sf); library(data.table); library(ggplot2); library(viridis); library(lubridate)
})

system(sprintf("cd /content && cp %s/saves_pilot_2026-06-03.tar.gz . && tar -xzf saves_pilot_2026-06-03.tar.gz", DRIVE))
load("/content/saves/pilot_session.RData")
source(file.path(DRIVE, "fns.R"))

year_start <- 2015; year_end <- 2019; warm_months <- 5:9
study_dates <- seq.Date(as.Date(sprintf("%d-05-01", year_start)),
                        as.Date(sprintf("%d-09-30", year_end)), by = "day")
study_dates <- study_dates[lubridate::month(study_dates) %in% warm_months]

system(sprintf("cd /content && rm -rf saves_eod && cp %s/saves_eod_2026-07-22.tar.gz . && tar -xzf saves_eod_2026-07-22.tar.gz", DRIVE))

red5  <- readRDS("/content/saves_eod/red5_v2.rds")
res5  <- readRDS("/content/saves_eod/res5_v2.rds")
Z5    <- readRDS("/content/saves_eod/Z5_v2.rds")
ids5  <- readRDS("/content/saves_eod/ids5_v2.rds")
diag5 <- readRDS("/content/saves_eod/diag5_v2.rds")
s2k5  <- readRDS("/content/saves_eod/stage2_k5_v2.rds")
cma_predictors <- readRDS("/content/saves_eod/cma_predictors.rds")

need <- c("read_daymet","build_crossbasis","make_strata_A","make_strata_B","qaic",
          "fit_city_sliver","build_city_sim_substrate_v2","simulate_counts",
          "fit_stage1","reduce_fit","fit_stage2","fit_da_pca","predict_da_theta",
          "compute_da_mmt","monte_carlo_ci","standardize_da_rate","save_to_drive")
fns_txt <- readLines(file.path(DRIVE, "fns.R"))
audit <- data.table(fn = need,
  in_env  = sapply(need, function(f) exists(f) && is.function(get(f))),
  in_file = sapply(need, function(f) any(grepl(sprintf("^%s <- function", f), fns_txt))))
audit[, landmine := in_env & !in_file]

cat("landmines:", sum(audit$landmine), " (stop if nonzero)\n")
cat("missing from file:", paste(audit[in_file == FALSE]$fn, collapse=", "), "\n")
cat("fns.R lines:", length(fns_txt), " (expect 406)\n")
cat("rds present:", length(list.files("/content/saves_eod")), " (expect 8)\n")
cat("study_dates:", length(study_dates), " L:", paste(dim(L), collapse=" x "),
    " DGP prims:", all(sapply(c("base_log_rr","lag_weights","L","da_age","annual_rates"), exists)), "\n")
cat("simulate_counts has exp:",
    grepl("exp(0.4", paste(deparse(body(simulate_counts)), collapse=" "), fixed = TRUE), "\n\n")

cat("stage2 coef length:", length(coef(s2k5$fit)), " (expect 20; if 5 the tarball is the K=3 fit)\n")
cat("stage2 formula:", deparse(s2k5$formula), "\n")
cat("theta lengths:", paste(sapply(red5, function(x) length(x$theta_star)), collapse=" "), "\n")
cat("cen per city:", paste(round(sapply(red5, function(x) x$cen), 2), collapse=" "), "\n")
cat("cb_template:", paste(dim(res5$Toronto$cb_template), collapse=" x "), "\n")
cat("Z rows:", paste(sapply(Z5, nrow), collapse=" "), " cols:", ncol(Z5$Toronto), "\n")
cat("ids match Z rows:", all(mapply(function(i, z) length(i) == nrow(z), ids5, Z5)), "\n")
print(diag5[, .(cma, n_da, deaths, ref_temp, winner)])

landmines: 0  (stop if nonzero)
missing from file:  
fns.R lines: 406  (expect 406)
rds present: 27  (expect 8)
study_dates: 765  L: 17 x 3  DGP prims: TRUE 
simulate_counts has exp: TRUE 

stage2 coef length: 20  (expect 20; if 5 the tarball is the K=3 fit)
stage2 formula: cbind(theta1, theta2, theta3, theta4, theta5) ~ PC1 + PC2 + PC3 
theta lengths: 5 5 5 5 5 
cen per city: 19.38 19.01 17.05 18.58 16.79 
cb_template: 114750 x 25 
Z rows: 7682 6504 3573 2042 1310  cols: 17 
ids match Z rows: TRUE 
         cma  n_da deaths ref_temp winner
      <char> <int>  <int>    <num> <char>
1:   Toronto  7682 610216 19.38333      A
2:  Montreal  6504 203988 19.00753      A
3: Vancouver  3573  56157 17.04646      A
4:    Ottawa  2042  51476 18.58291      A
5:    Quebec  1310  26977 16.79456      A


landmines: 0  (stop if nonzero)
missing from file:  
fns.R lines: 406  (expect 406)
rds present: 27  (expect 8)
study_dates: 765  L: 17 x 3  DGP prims: TRUE
simulate_counts has exp: TRUE

stage2 coef length: 20  (expect 20; if 5 the tarball is the K=3 fit)
stage2 formula: cbind(theta1, theta2, theta3, theta4, theta5) ~ PC1 + PC2 + PC3
theta lengths: 5 5 5 5 5
cen per city: 19.38 19.01 17.05 18.58 16.79
cb_template: 114750 x 25
Z rows: 7682 6504 3573 2042 1310  cols: 17
ids match Z rows: TRUE
         cma  n_da deaths ref_temp winner
      <char> <int>  <int>    <num> <char>
1:   Toronto  7682 610216 19.38333      A
2:  Montreal  6504 203988 19.00753      A
3: Vancouver  3573  56157 17.04646      A
4:    Ottawa  2042  51476 18.58291      A
5:    Quebec  1310  26977 16.79456      A

---

§0 restore clean. landmines 0, fns.R 406, dgp prims live, simulate_counts is the exp form.
coef length 20 and formula ~PC1+PC2+PC3 read. cen 19.38 19.01 17.05 18.58 16.79, theta lengths all 5, cb_template
114750x25, Z rows 7682/6504/3573/2042/1310 with ids matching. every number reproduces
07-22 to the digit

list the 27 and check whether a toronto substrate is
in there. same call checks the geojson §8.8
reads for the choropleth.

In [ ]:
f <- list.files("/content/saves_eod", full.names = TRUE)
inv <- data.table(obj = sub("\\.rds$", "", basename(f)),
                  mb  = round(file.size(f) / 1024^2, 2))
setorder(inv, -mb)

cat("files:", nrow(inv), "\n\n")
print(inv)

cat("\n--- toronto-shaped ---\n")
print(grep("tor|truth|substrate", inv$obj, ignore.case = TRUE, value = TRUE))

cat("\ntotal mb:", round(sum(inv$mb), 1), "\n")
cat("tarball mb:", round(file.size(file.path(DRIVE, "saves_eod_2026-07-22.tar.gz")) / 1024^2, 1), "\n")

cat("\n--- polygons on drive ---\n")
g <- list.files(DRIVE, pattern = "toronto.*(geojson|shp\\.zip)$", full.names = TRUE)
print(data.table(file = basename(g), mb = round(file.size(g) / 1024^2, 2)))

files: 27 

                    obj    mb
                 <char> <num>
 1:             res5_v2 87.93
 2:       mtl_substrate 20.26
 3:             red_tor 17.75
 4:         red_mtl_fit 17.65
 5:         red_van_fit 17.34
 6:       van_substrate 12.63
 7:               Z5_v2  2.62
 8:               Z3_v2  2.21
 9: cma_age_data_mtlvan  0.09
10:             ids5_v2  0.05
11:             ids3_v2  0.04
12:       new_age_ottqc  0.03
13:             red5_v2  0.02
14:        stage2_k5_v2  0.02
15:             red3_v2  0.01
16:      cma_predictors  0.00
17:            diag3_v2  0.00
18:            diag5_v2  0.00
19:       fn_fit_stage1  0.00
20:             fn_qaic  0.00
21:       fn_reduce_fit  0.00
22:          reduced_df  0.00
23:         reduced_mtl  0.00
24:         reduced_tor  0.00
25:         reduced_van  0.00
26:              stage2  0.00
27:           vcov_list  0.00
                    obj    mb
                 <char> <num>

--- toronto-shaped ---
[1] "mtl_substrate"  "red_tor"    

files: 27

                    obj    mb
                 <char> <num>
 1:             res5_v2 87.93
 2:       mtl_substrate 20.26
 3:             red_tor 17.75
 4:         red_mtl_fit 17.65
 5:         red_van_fit 17.34
 6:       van_substrate 12.63
 7:               Z5_v2  2.62
 8:               Z3_v2  2.21
 9: cma_age_data_mtlvan  0.09
10:             ids5_v2  0.05
11:             ids3_v2  0.04
12:       new_age_ottqc  0.03
13:             red5_v2  0.02
14:        stage2_k5_v2  0.02
15:             red3_v2  0.01
16:      cma_predictors  0.00
17:            diag3_v2  0.00
18:            diag5_v2  0.00
19:       fn_fit_stage1  0.00
20:             fn_qaic  0.00
21:       fn_reduce_fit  0.00
22:          reduced_df  0.00
23:         reduced_mtl  0.00
24:         reduced_tor  0.00
25:         reduced_van  0.00
26:              stage2  0.00
27:           vcov_list  0.00
                    obj    mb
                 <char> <num>

--- toronto-shaped ---
[1] "mtl_substrate"  "red_tor"        "van_substrate"  "cma_predictors"
[5] "reduced_tor"   

total mb: 178.7
tarball mb: 178.5

--- polygons on drive ---
                 file    mb
               <char> <num>
1: toronto_da_shp.zip  4.92
2: toronto_da.geojson 37.82

---



toronto substrate at seed 42+35=77 for truth_factors + temp_mat.

mu, predict [-0.330, 0.655, 0.384] — same seed same L same da_age so the rnorm(3,0,0.6)
draw reproduces. wrong mu = wrong dgp draw and every shape check still passes.

n_da 7682 matching diag5, temp_mat 7682x765, temp NA 0, rownames aligned to truth_factors$DAUID.
truth sd 0.4472 = sqrt(0.16+0.04), mu doesn't enter an sd so it reproduces v1's 0.447. truth
mean 0.945 not v1's 1.00 — that's where the offset shows. range about -0.8 to 2.7.

In [ ]:
dm  <- read_daymet(file.path(DRIVE, "toronto_daymet_2015_2019.csv"))
cat("daymet rows:", nrow(dm), " DAs:", uniqueN(dm$DAUID), " dates:", uniqueN(dm$date), "\n")

sub <- build_city_sim_substrate_v2(dm, da_age, L, seed = 42 + 35)
rm(dm); gc(verbose = FALSE)

temp_mat      <- sub$temp_mat
truth_factors <- sub$truth_factors

cat("\nmu:", paste(round(sub$mu, 3), collapse=" "), "\n")
cat("n_da:", sub$n_da, " (expect 7682)\n")
cat("temp_mat:", paste(dim(temp_mat), collapse=" x "), " NA:", sum(is.na(temp_mat)), "\n")
cat("temp range:", paste(round(range(temp_mat), 2), collapse=" "), "\n")
cat("truth_factors:", paste(dim(truth_factors), collapse=" x "), "\n")
cat("DAUID is chr:", is.character(truth_factors$DAUID), "\n")
cat("aligned:", all(rownames(temp_mat) == truth_factors$DAUID), "\n")
cat("ids5 Toronto match:", all(sort(ids5$Toronto) == sort(truth_factors$DAUID)), "\n")

truth_vuln <- data.table(DAUID = truth_factors$DAUID,
                         truth = 1 + 0.4 * truth_factors$F1 + 0.2 * truth_factors$F3)

cat("\ntruth sd:", round(sd(truth_vuln$truth), 4), " (expect 0.4472)\n")
cat("truth mean:", round(mean(truth_vuln$truth), 3), " (expect 0.945)\n")
cat("truth range:", paste(round(range(truth_vuln$truth), 2), collapse=" "), "\n")
cat("mem gb:", round(sum(gc()[,2])/1024, 2), "\n")

daymet rows: 5902740  DAs: 7716  dates: 765 


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2641231,141.1,5155321,275.4,5155321,275.4
Vcells,145068800,1106.8,221688089,1691.4,221683934,1691.4



mu: -0.33 0.655 0.384 
n_da: 7682  (expect 7682)
temp_mat: 7682 x 765  NA: 0 
temp range: 2.2 30.15 
truth_factors: 7682 x 5 
DAUID is chr: TRUE 
aligned: TRUE 
ids5 Toronto match: TRUE 

truth sd: 0.4454  (expect 0.4472)
truth mean: 0.946  (expect 0.945)
truth range: -0.61 2.92 
mem gb: 1.22 


daymet rows: 5902740  DAs: 7716  dates: 765
A matrix: 2 × 6 of type dbl
used	(Mb)	gc trigger	(Mb)	max used	(Mb)
Ncells	2641231	141.1	5155321	275.4	5155321	275.4
Vcells	145068800	1106.8	221688089	1691.4	221683934	1691.4

mu: -0.33 0.655 0.384
n_da: 7682  (expect 7682)
temp_mat: 7682 x 765  NA: 0
temp range: 2.2 30.15
truth_factors: 7682 x 5
DAUID is chr: TRUE
aligned: TRUE
ids5 Toronto match: TRUE

truth sd: 0.4454  (expect 0.4472)
truth mean: 0.946  (expect 0.945)
truth range: -0.61 2.92
mem gb: 1.22

---



predict_da_theta verbatim against the k=5 pool, then §8.8 verbatim, two renames (res5$Toronto$cb_template, red5$Toronto$cen) and the five
intercepts sliced by name out of the 20. no map render, the flat panel already exists.

predict uniqueN(recovered_rr) = 1 — one scalar recycled 7682 times, so sd is exactly 0 not
1e-16. truth sd 0.4454 carried from the last block. heat_p about 28.
rr_flat >1, order 10, estimate not a digit

In [ ]:
cf20 <- coef(s2k5$fit)
cat("coef length:", length(cf20), "\n\n")

attempt <- try(predict_da_theta(s2k5$fit, truth_factors$DAUID), silent = TRUE)
cat("§8.2 verbatim status:", if (inherits(attempt, "try-error")) "RAISED" else "ran", "\n")
if (inherits(attempt, "try-error")) cat("  ->", conditionMessage(attr(attempt, "condition")), "\n")

int_idx <- grep("\\(Intercept\\)$", names(cf20))
cf <- as.numeric(cf20[int_idx])
cat("\nintercepts by name:", length(int_idx), " names:", paste(names(cf20)[int_idx], collapse=" "), "\n")
cat("cf:", paste(round(cf, 3), collapse=" "), "\n")

av        <- attr(res5$Toronto$cb_template, "argvar")
cen_tor   <- red5$Toronto$cen
temp_grid <- seq(min(temp_mat), max(temp_mat), length.out = 100)
red_basis <- onebasis(temp_grid, fun = av$fun, degree = av$degree, knots = av$knots)

cat("\nargvar fun:", av$fun, " degree:", av$degree, " knots:", paste(round(av$knots, 2), collapse=" "), "\n")
cat("red_basis:", paste(dim(red_basis), collapse=" x "), " (need 100 x 5)\n")
stopifnot(ncol(red_basis) == length(cf))

pred_flat <- crosspred(basis = red_basis, coef = cf, vcov = red5$Toronto$V_star,
                       at = temp_grid, cen = cen_tor)

heat_p  <- quantile(temp_mat, 0.99)
rr_flat <- approx(pred_flat$predvar, pred_flat$allRRfit, xout = heat_p)$y
recovered <- data.table(DAUID = truth_factors$DAUID, recovered_rr = rr_flat)

cat("\ncen:", round(cen_tor, 2), " heat_p:", round(heat_p, 2), "\n")
cat("RR at cen:", round(approx(pred_flat$predvar, pred_flat$allRRfit, xout = cen_tor)$y, 4), " (must be 1)\n")
cat("RR range over grid:", paste(round(range(pred_flat$allRRfit), 3), collapse=" "), "\n")
cat("rr_flat:", round(rr_flat, 4), "\n\n")

cat("recovered rows:", nrow(recovered), " distinct values:", uniqueN(recovered$recovered_rr), "\n")
cat("recovered sd:", sprintf("%.17g", sd(recovered$recovered_rr)), "\n")
cat("sd identical to 0:", identical(sd(recovered$recovered_rr), 0), "\n")
cat("truth sd:", round(sd(truth_vuln$truth), 4), " mean:", round(mean(truth_vuln$truth), 3), "\n")
cat("exp target sd:", round(sd(exp(0.4*truth_factors$F1 + 0.2*truth_factors$F3)), 4), "\n")

coef length: 20 

§8.2 verbatim status: RAISED 
  -> length(cf) == 5 is not TRUE 

intercepts by name: 5  names: theta1.(Intercept) theta2.(Intercept) theta3.(Intercept) theta4.(Intercept) theta5.(Intercept) 
cf: -0.872 -4.606 -4.481 -3.813 -1.02 

argvar fun: bs  degree: 2  knots: 12.51 22.18 24.27 
red_basis: 100 x 5  (need 100 x 5)

cen: 19.38  heat_p: 27.19 
RR at cen: 7.3934  (must be 1)


Warning message in min(x, na.rm = na.rm):
“no non-missing arguments to min; returning Inf”
Warning message in max(x, na.rm = na.rm):
“no non-missing arguments to max; returning -Inf”


RR range over grid: Inf -Inf 
rr_flat: 9.5955 

recovered rows: 7682  distinct values: 1 
recovered sd: 0 
sd identical to 0: TRUE 
truth sd: 0.4454  mean: 0.946 
exp target sd: 0.5002 


crosspred with raw coef/vcov and no model.link returns allfit only — no allRRfit. fix is one argument, model.link = "log". §7.1 already says it's required with raw coef/vcov,
reduce_fit passes it, §8.8 didn't.

predict RR at cen exactly 1 read first. RR at the cold end ~80 — bs drops its first column so
allfit(min)=0 by construction and RR(min) = exp(-allfit(cen)) = exp(4.4). rr_flat ~10, range
5-20. rr 10 for heat isn't physical, same dgp inflation as lambda max 9759.

In [ ]:
pred_log <- crosspred(basis = red_basis, coef = cf, vcov = red5$Toronto$V_star,
                      model.link = "log", at = temp_grid, cen = cen_tor)

cat("allRRfit present:", !is.null(pred_log$allRRfit),
    " length:", length(pred_log$allRRfit), "\n")

rr_cen <- approx(pred_log$predvar, pred_log$allRRfit, xout = cen_tor)$y
cat("RR at cen:", round(rr_cen, 6), " (must be 1)\n")
cat("RR range:", paste(round(range(pred_log$allRRfit), 2), collapse=" "), "\n")
cat("RR at cold end:", round(pred_log$allRRfit[1], 2), " at hot end:",
    round(pred_log$allRRfit[100], 2), "\n")

manual <- as.vector(red_basis %*% cf)
i_cen  <- which.min(abs(temp_grid - cen_tor))
cat("manual allfit at min:", round(manual[1], 6), " (bs drops col 1 -> expect 0)\n")
cat("manual vs crosspred max abs diff:",
    signif(max(abs(exp(manual - manual[i_cen]) - pred_log$allRRfit)), 3), "\n")

rr_flat <- approx(pred_log$predvar, pred_log$allRRfit, xout = heat_p)$y
recovered <- data.table(DAUID = truth_factors$DAUID, recovered_rr = rr_flat)

cat("\nheat_p:", round(heat_p, 2), "  rr_flat:", round(rr_flat, 3), "\n")
cat("recovered rows:", nrow(recovered), " distinct:", uniqueN(recovered$recovered_rr),
    " sd:", sd(recovered$recovered_rr), "\n")
cat("mmt (grid argmin):", round(temp_grid[which.min(pred_log$allRRfit)], 2), "\n")

allRRfit present: TRUE  length: 100 
RR at cen: 1.000241  (must be 1)
RR range: 0.9 81.68 
RR at cold end: 81.68  at hot end: 29.46 
manual allfit at min: 0  (bs drops col 1 -> expect 0)
manual vs crosspred max abs diff: 0.298 

heat_p: 27.19   rr_flat: 3.176 
recovered rows: 7682  distinct: 1  sd: 0 
mmt (grid argmin): 21.68 
